In [9]:
!pip install pageindex openai


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import asyncio

from pageindex import PageIndexClient
import pageindex.utils as utils

In [10]:
PAGEINDEX_API_KEY = os.environ.get("PAGEINDEX_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if not PAGEINDEX_API_KEY:
    raise ValueError("Missing PAGEINDEX_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("Missing OPENAI_API_KEY")

In [4]:
# Create the PageIndex client once
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [11]:
async def call_llm(prompt: str, model: str = "gpt-4.1", temperature: float = 0):
    import openai

    client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)

    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )

    return response.choices[0].message.content.strip()

In [ ]:
async def deepseek_demo():

    pdf_path = os.path.abspath("Nvidia.pdf")

    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Could not find PDF at: {pdf_path}")

    # Submit document
    doc_id = pi_client.submit_document(pdf_path)["doc_id"]
    print("Document submitted:", doc_id)

    # Wait until processing finishes
    while not pi_client.is_retrieval_ready(doc_id):
        print("Waiting for processing...")
        await asyncio.sleep(2)

    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]

    print("Generated tree:")
    utils.print_tree(tree)

    # Query
    query = "Stem Field?"

    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    search_prompt = f"""
You are given a question and a tree structure of a document.

Each node contains a node id, node title, and summary.

Find nodes that likely contain the answer.

Question: {query}

Tree:
{json.dumps(tree_without_text, indent=2)}

Return JSON:

{{
 "thinking":"reasoning",
 "node_list":["node1","node2"]
}}
"""

    tree_search_result = await call_llm(search_prompt)

    node_map = utils.create_node_mapping(tree)

    tree_search_result_json = json.loads(tree_search_result)

    print("\nReasoning:")
    utils.print_wrapped(tree_search_result_json["thinking"])

    print("\nRetrieved Nodes:")

    for nid in tree_search_result_json["node_list"]:
        node = node_map[nid]

        print(
            f"Node ID: {node['node_id']} | "
            f"Page: {node.get('page_index')} | "
            f"Title: {node['title']}"
        )

    # Extract text
    relevant_text = "\n\n".join(
        node_map[nid]["text"]
        for nid in tree_search_result_json["node_list"]
        if nid in node_map and "text" in node_map[nid]
    )

    print("\nContext snippet:\n")
    utils.print_wrapped(relevant_text[:1000] + "...")

    answer_prompt = f"""
Answer the question based on the context.

Question: {query}

Context:
{relevant_text}
"""

    answer = await call_llm(answer_prompt)

    print("\nFinal Answer:\n")
    utils.print_wrapped(answer)

RuntimeError: asyncio.run() cannot be called from a running event loop